# PerNeuronDropDense — CIFAR-10 Experiments

This notebook benchmarks the `PerNeuronDropDense` layer against a standard
`Dense + Dropout` baseline on CIFAR-10, using a small CNN feature extractor
and swapping only the classifier head.

Configurations compared:

1. **Baseline** — `Dense` + `Dropout`
2. **PerNeuronDropDense (drop)** — Bernoulli connection masking
3. **PerNeuronDropDense (gaussian)** — multiplicative Gaussian weight noise
4. **PerNeuronDropDense (fixed mask)** — a single frozen random sub-network

Each is trained for a few epochs on CIFAR-10 and compared on validation
accuracy/loss curves and final test accuracy.


In [ ]:
import os
import sys
import time

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Make the layer importable whether the notebook is run from the repo root
# or from inside notebook/
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

from per_neuron_drop_dense import PerNeuronDropDense

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)


## 1. Load and preprocess CIFAR-10

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.squeeze()
y_test = y_test.squeeze()

# Hold out a validation split from the training set
val_split = 5000
x_val, y_val = x_train[-val_split:], y_train[-val_split:]
x_train, y_train = x_train[:-val_split], y_train[:-val_split]

class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

print("train:", x_train.shape, "val:", x_val.shape, "test:", x_test.shape)


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[y_train[i]], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Shared CNN feature extractor + swappable classifier head

All configurations share the same convolutional backbone; only the dense
classifier head changes, so any difference in results is attributable to
the head layer being tested.


In [ ]:
def build_backbone():
    """Convolutional feature extractor shared by every configuration."""
    inputs = tf.keras.Input(shape=(32, 32, 3))
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.Conv2D(32, 3, activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv2D(64, 3, activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Flatten()(x)
    return inputs, x


def build_model(head_fn, name):
    inputs, features = build_backbone()
    x = head_fn(features)
    outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name=name)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# --- Classifier heads under comparison -------------------------------

def head_baseline_dropout(x, drop_rate=0.3):
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(drop_rate)(x)
    return x


def head_drop(x, drop_rate=0.3):
    return PerNeuronDropDense(256, drop_rate=drop_rate, stir_type="drop",
                               activation="relu")(x)


def head_gaussian(x, drop_rate=0.3):
    return PerNeuronDropDense(256, drop_rate=drop_rate, stir_type="gaussian",
                               activation="relu")(x)


def head_fixed_mask(x, drop_rate=0.3):
    return PerNeuronDropDense(256, drop_rate=drop_rate, stir_type="drop",
                               fixed_mask=True, activation="relu")(x)


CONFIGS = {
    "baseline_dropout": head_baseline_dropout,
    "per_neuron_drop": head_drop,
    "per_neuron_gaussian": head_gaussian,
    "per_neuron_fixed_mask": head_fixed_mask,
}


## 3. Train each configuration

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

histories = {}
test_scores = {}

for name, head_fn in CONFIGS.items():
    print(f"\n=== Training: {name} ===")
    tf.random.set_seed(42)
    model = build_model(head_fn, name=name)

    start = time.time()
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=2,
    )
    elapsed = time.time() - start

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"{name}: test_acc={test_acc:.4f}  test_loss={test_loss:.4f}  ({elapsed:.1f}s)")

    histories[name] = history.history
    test_scores[name] = {"test_acc": test_acc, "test_loss": test_loss, "train_time_s": elapsed}


## 4. Compare validation curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for name, hist in histories.items():
    axes[0].plot(hist["val_accuracy"], label=name)
    axes[1].plot(hist["val_loss"], label=name)

axes[0].set_title("Validation accuracy")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("accuracy")
axes[0].legend()

axes[1].set_title("Validation loss")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("loss")
axes[1].legend()

plt.tight_layout()
plt.show()


## 5. Final test-set comparison

In [ ]:
import pandas as pd

results_df = pd.DataFrame(test_scores).T.sort_values("test_acc", ascending=False)
results_df


In [ ]:
results_df["test_acc"].plot(kind="bar", figsize=(7, 4), rot=20, title="Test accuracy by configuration")
plt.ylabel("test accuracy")
plt.tight_layout()
plt.show()


## 6. Drop-rate sweep (optional)

A quick sweep over `drop_rate` for the `stir_type="drop"` variant, to see
how the regularization strength trades off against accuracy.


In [ ]:
drop_rates = [0.1, 0.3, 0.5]
sweep_results = {}

for dr in drop_rates:
    name = f"per_neuron_drop_rate_{dr}"
    print(f"\n=== Training: {name} ===")
    tf.random.set_seed(42)
    model = build_model(lambda x, dr=dr: head_drop(x, drop_rate=dr), name=name)
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
    )
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"drop_rate={dr}: test_acc={test_acc:.4f}")
    sweep_results[dr] = test_acc

plt.figure(figsize=(6, 4))
plt.plot(list(sweep_results.keys()), list(sweep_results.values()), marker="o")
plt.xlabel("drop_rate")
plt.ylabel("test accuracy")
plt.title("Effect of drop_rate on test accuracy")
plt.show()


## Notes

- All configurations share the same convolutional backbone and are trained
  with the same optimizer, batch size, and number of epochs, so differences
  in the curves above are attributable to the classifier head.
- `fixed_mask=True` freezes a single random sub-network for the entire
  training run — expect it to behave more like a smaller, fixed-capacity
  network than a regularizer that changes every step.
- Try increasing `EPOCHS`, adding data augmentation, or testing the layer
  at different positions in the network for more thorough experiments.
